In [1]:
from pyspark.sql import SparkSession
import time

In [2]:
spark = SparkSession.builder \
    .appName("Spark_Lab") \
    .master("spark://spark-master:7077") \
    .config("spark.ui.port", "4043") \
    .config("spark.executor.instances","2") \
    .config("spark.executor.core","1") \
    .config("spark.executor.memory","1g") \
    .config("spark.cores.max", "2") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 03:30:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext
rdd = sc.textFile("/opt/spark/work-dir/data/orders.csv",4)

In [7]:
print(rdd.count())

26/05/09 12:38:51 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/09 12:39:06 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/05/09 12:39:21 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
[Stage 0:>                                                          (0 + 2) / 4]

2001


In [4]:
header = rdd.first()

In [5]:
print(header)

order_id,customer_id,city,product,category,quantity,price


In [6]:
rdd_no_header = rdd.filter(lambda x: x != header)

In [14]:
print(rdd_no_header.take(5))

['1,C041,CanTho,Desk,Furniture,4,300', '2,C216,HCM,Phone,Electronics,2,900', '3,C172,Hanoi,Keyboard,Electronics,1,80', '4,C250,HCM,Phone,Electronics,5,900', '5,C215,Hanoi,Chair,Furniture,3,120']


In [7]:
rdd_split = rdd_no_header.map(lambda x: x.split(','))

In [16]:
print(rdd_split.take(2))

[['1', 'C041', 'CanTho', 'Desk', 'Furniture', '4', '300'], ['2', 'C216', 'HCM', 'Phone', 'Electronics', '2', '900']]


In [18]:
quantity_order = rdd_split.map(lambda x: (x[1], int(x[5]))).reduceByKey(lambda a,b: a + b).sortBy(lambda x: x[1], ascending=False)

In [22]:
print(quantity_order.take(5))

[('C275', 60), ('C059', 42), ('C122', 39), ('C170', 39), ('C045', 39)]


Practice 12

In [26]:
revenue_by_city = rdd_split.map(lambda x: (x[2], int(x[5])*int(x[6]))).reduceByKey(lambda a,b: a + b)

In [27]:
max_revenue_city = revenue_by_city.max(key=lambda x: x[1])

In [29]:
print(max_revenue_city)

('CanTho', 610515)


In [25]:
print(revenue_by_city.take(5))

[('CanTho', 1200), ('HCM', 1800), ('Hanoi', 80), ('HCM', 4500), ('Hanoi', 360)]


Practice 13

In [32]:
highest_quantity_product = rdd_split.map(lambda x: (x[3],int(x[5]))).reduceByKey(lambda a,b: a+b).max(key=lambda x: x[1])

In [33]:
print(highest_quantity_product)

('Phone', 720)


Practice 14

In [8]:
rdd_city = rdd_split.map(
    lambda x: (
        x[2], (int(x[5])*int(x[6]),1)
    )
)

In [9]:
rdd_sum_revunue_quantity = rdd_city.reduceByKey(
    lambda a,b: (
        a[0] + b[0],
        a[1] + b[1]
    )
)

In [10]:
rdd_avg_rev_by_city = rdd_sum_revunue_quantity.map(
    lambda x: (
        x[0] , round(x[1][0] / x[1][1],2)
    )
)

In [11]:
rdd_avg_rev_by_city.cache()

PythonRDD[7] at RDD at PythonRDD.scala:53

In [12]:
print(rdd_avg_rev_by_city.collect())

[Stage 1:============================================>              (3 + 1) / 4]

[('HCM', 1089.55), ('CanTho', 1218.59), ('Danang', 1183.76), ('Hanoi', 1092.06)]


In [13]:
print(rdd_avg_rev_by_city.count())

4
